In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class DigitClassifierNet(nn.Module):
    def __init__(self):
        super(DigitClassifierNet, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 10) 
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x

def train_mnist_model(model, device, epochs=10):
    print(f"Training neural network on MNIST dataset for {epochs} epochs...")
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])
    
    train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
    train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        correct_predictions = 0
        total_samples = 0
        
        for data, target in train_loader:
            data, target = data.to(device), target.to(device)
            
            # Forward pass
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
            predictions = output.argmax(dim=1)
            correct_predictions += (predictions == target).sum().item()
            total_samples += target.size(0)
            
        avg_loss = running_loss / len(train_loader)
        epoch_accuracy = (correct_predictions / total_samples) * 100
            
        print(f"Epoch {epoch+1:02d}/{epochs} completed | Avg Loss: {avg_loss:.4f} | Accuracy: {epoch_accuracy:.2f}%")
    
    torch.save(model.state_dict(), "mnist_cnn.pth")
    print("Model trained successfully and saved as 'mnist_cnn.pth'\n")

class ImageFolderDataset(Dataset):
    def __init__(self, folder_path, transform=None):
        self.folder_path = folder_path
        self.transform = transform
        valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp')
        self.filenames = sorted([f for f in os.listdir(folder_path) if f.lower().endswith(valid_extensions)])

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        filename = self.filenames[idx]
        filepath = os.path.join(self.folder_path, filename)
        try:
            img = Image.open(filepath).convert('L')
            if self.transform:
                img = self.transform(img)
            return img, 1  
        except Exception:
            return torch.zeros(1, 28, 28), 0  


def count_digits_in_folder(folder_path):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    model = DigitClassifierNet().to(device)
    
    if not os.path.exists("mnist_cnn.pth"):
        train_mnist_model(model, device, epochs=15)
    else:
        print("Loading pre-trained weights from 'mnist_cnn.pth'...")
        model.load_state_dict(torch.load("mnist_cnn.pth", map_location=device))
    
    model.eval()

    preprocess = transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.Resize((28, 28)),
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])

    # Instantiate dataset and batch loader
    dataset = ImageFolderDataset(folder_path, transform=preprocess)
    dataloader = DataLoader(dataset, batch_size=256, shuffle=False, num_workers=0)

    digit_counts = [0] * 10
    total_images = len(dataset)
    processed_count = 0
    corrupted_count = 0

    print(f"Starting batch inference on {total_images} images...")

    with torch.no_grad():
        for imgs, valid_flags in dataloader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            predictions = outputs.argmax(dim=1)

            # Accumulate predictions based on validity flags
            for pred, is_valid in zip(predictions, valid_flags):
                processed_count += 1
                if is_valid == 1:
                    digit_counts[pred.item()] += 1
                else:
                    corrupted_count += 1

            # Print concise progress updates to prevent flooding the terminal
            if processed_count % 1024 == 0 or processed_count == total_images:
                print(f"Progress: {processed_count}/{total_images} files processed...")

    if corrupted_count > 0:
        print(f"Warning: Bypassed {corrupted_count} corrupted image file(s).")

    return digit_counts

if __name__ == "__main__":
    image_folder = "digits/" 
    
    if not os.path.exists(image_folder):
        os.makedirs(image_folder)
        print(f"Created directory '{image_folder}'. Please put your images inside.")
    else:
        final_counts = count_digits_in_folder(image_folder)
        print("\nFinal 10-Element Array [0_count, 1_count, ..., 9_count]:")
        print(final_counts)

Using device: cuda
Training neural network on MNIST dataset for 15 epochs...
Epoch 01/15 completed | Avg Loss: 0.1893 | Accuracy: 94.18%
Epoch 02/15 completed | Avg Loss: 0.0547 | Accuracy: 98.33%
Epoch 03/15 completed | Avg Loss: 0.0387 | Accuracy: 98.81%
Epoch 04/15 completed | Avg Loss: 0.0295 | Accuracy: 99.06%
Epoch 05/15 completed | Avg Loss: 0.0231 | Accuracy: 99.27%
Epoch 06/15 completed | Avg Loss: 0.0194 | Accuracy: 99.40%
Epoch 07/15 completed | Avg Loss: 0.0168 | Accuracy: 99.40%
Epoch 08/15 completed | Avg Loss: 0.0146 | Accuracy: 99.51%
Epoch 09/15 completed | Avg Loss: 0.0127 | Accuracy: 99.61%
Epoch 10/15 completed | Avg Loss: 0.0111 | Accuracy: 99.62%
Epoch 11/15 completed | Avg Loss: 0.0093 | Accuracy: 99.68%
Epoch 12/15 completed | Avg Loss: 0.0079 | Accuracy: 99.73%
Epoch 13/15 completed | Avg Loss: 0.0091 | Accuracy: 99.67%
Epoch 14/15 completed | Avg Loss: 0.0075 | Accuracy: 99.76%
Epoch 15/15 completed | Avg Loss: 0.0075 | Accuracy: 99.72%
Model trained successfu